## Legenda delle funzioni e della classe Planet

#### Funzioni
- **initial_conditions** : Permette di assegnare le condizioni iniziali del sistema
- **update** : aggiorna le posizioni, velocità ed accelerazioni con il metodo di Eulero
- **Set_Range** : Permette di trovare il range da assegnare ad entrambi gli assi del grafico affinchè le orbite ellittiche rimangano tali e non vengano rappresentate graficamente come circolari (a causa di un uso di diverse scale su x e y).
- **eccentricity**: Calcola il valore dell'eccentricità dell'orbita a partire dalle condizioni iniziali.
- **Second_Law** : Calcola il valore del momento angolare specifico per P_1, P_2, P_rel.
- **Third_Law**: Calcola il periodo e il semiasse maggiore, calcolando la costante di Keplero K.
#### Classe Planet
- **self.m** : massa
- **self.x** : coordinata x
- **self.y** : coordinata y
- **self.vX** : velocità lungo x
- **self.vY** : velocità lungo y
- **self.accX** : accelerazione lungo x
- **self.accY** : accelerazione lungo y
- **self.h** : momento angolare specifico

<hr style="border:2px solid gray">

## Suggerimenti dati iniziali

####             Sistema Sole-Mercurio 
V_in= 59000       
r =46000000000         
m1=  3.285 * pow(10,23)       
m2= 1.989 * pow(10,30)     
dt=5          
n_cycles=2000000       

#### Sistema Sole-Terra 
V_in= 29300        
r = 152100000000       
m1=  5.97 * pow(10,24)     
m2= 1.989 * pow(10,30)    
dt=18    
n_cycles=2000000 

#### Sistema Sole-Marte 
V_in= 21970       
r = 249000000000       
m1=  6.4 * pow(10,23)     
m2= 1.989 * pow(10,30)  
dt=30                          
n_cycles=2000000  

#### Sistema Sole-Giove
V_in= 13710            
r = 740000000000            
m1= 1.9 * pow(10,27)            
m2= 1.989 * pow(10,30)            
dt=190                 
n_cycles=2000000            


#### Sistema Sole-Cometa 2P/Encke (Molto soggetta ad approssimazione durante il transito nel perielio)
V_in=3700        
r= 612453683000        
m1= 1* pow(10,14)       
m2= 1.989 * pow(10,30)     
dt=55        
n_cycles=2000000

#### Sistema Terra-Luna 
V_in=1076        
r = 363000000                  
m1 = 7.342 * pow(10,22)     
m2 = 5.972 * pow(10,24)      
dt=1.5      
n_cycles=2000000 

<hr style="border:2px solid gray">     

## Simulazione

In [1]:
import numpy as np
import ROOT
import math
import array

Error=0
G= 6.67 * pow(10,-11)

In [2]:
#Sistema Sole-Terra:
V_in= 29300
r = 152100000000
m1= 5.97 * pow(10,24)
m2= 1.989 * pow(10,30)
dt=18
n_cycles=2000000


Total_Simulated_Time= (dt*n_cycles)/60/60/24
print(f"Saranno simulati {Total_Simulated_Time:.2f} giorni terrestri, equivalenti a {(Total_Simulated_Time/365):.2f} anni")

Saranno simulati 416.67 giorni terrestri, equivalenti a 1.14 anni


In [3]:
class Planet:
    def __init__(self, m):
        self.m = m
        
        self.x =[]
        self.y=[]
        
        self.vX=[]
        self.vY=[]
        
        self.accX =[]
        self.accY =[]

        self.h = []

In [4]:
P_1 = Planet(m1)
P_2= Planet(m2)
P_rel=Planet(m1*m2/(m1+m2))

In [5]:
def initial_conditions(P_1,P_2,P_rel,V_in, r):

    P_1.x= [r * (P_2.m/(P_1.m+P_2.m))]
    P_1.y= [0]
    P_2.x= [r * (-P_1.m/(P_1.m+P_2.m))]
    P_2.y= [0]
    P_1.vX = [0]
    P_1.vY = [V_in * (P_2.m/(P_1.m+P_2.m))]
    P_2.vX = [0]
    P_2.vY = [V_in * (-P_1.m/(P_1.m+P_2.m))]
    
    P_rel.x = [r]
    P_rel.y = [0]
    P_rel.vX = [0]
    P_rel.vY = [V_in]
    P_rel.accX = []
    P_rel.accY= []

**Funzione Update**       
Utilizzo il metodo di riduzione del problema ad un solo corpo.        
Posso studiare il moto del corpo "relativo" utilizzando il metodo di Eulero per calcolare successivamente le posizioni e velocità dei due corpi.
L'accelerazione può essere calcolata a partire da:
- $a_{x,rel} = - \frac{G(m_1 + m_2)x}{(x^2 + y^2)^{3/2}} $
- $a_{y,rel} = - \frac{G(m_1 + m_2)y}{(x^2 + y^2)^{3/2}}$
  
Per passare alle posizioni dei due corpi ricordo che:                    
- $\vec{r}_1 = \frac{m_2}{m_1 + m_2} \vec{r} $                       
- $\vec{r}_2 = -\frac{m_1}{m_1 + m_2} \vec{r}      $           

e analogamente con le velocità:                    
- $\vec{v}_1 = \frac{m_2}{m_1 + m_2} \vec{v}_{rel}$             
- $\vec{v}_2 = -\frac{m_1}{m_1 + m_2} \vec{v}_{rel}$


In [6]:
def update(P_1,P_2,P_rel,dt,j):
     
    #aggiornamento accelerazioni
    P_rel.accX.append ( -P_rel.x[j]/pow(pow(P_rel.x[j],2)+pow(P_rel.y[j],2),(3/2)) * G * (P_1.m+P_2.m))
    P_rel.accY.append ( -P_rel.y[j]/pow(pow(P_rel.x[j],2)+pow(P_rel.y[j],2),(3/2)) * G * (P_1.m+P_2.m))

    # Calcolo le posizioni e velocità di indice j+1
    
    #aggiornamento posizioni
    P_rel.x.append( P_rel.x[j] + (P_rel.vX[j]*dt) + (P_rel.accX[j]/2)*pow(dt,2))
    P_rel.y.append( P_rel.y[j] +(P_rel.vY[j]*dt) + (P_rel.accY[j]/2)*pow(dt,2))
    
    #aggiornamento velocità
    P_rel.vX.append( P_rel.vX[j] + P_rel.accX[j]*dt)
    P_rel.vY.append( P_rel.vY[j] + P_rel.accY[j]*dt)
    
    #coordinate e velocità  dei due corpi
    P_1.x.append( P_rel.x[-1]*(P_2.m/(P_1.m+P_2.m)))
    P_1.y.append( P_rel.y[-1]*(P_2.m/(P_1.m+P_2.m)) )
    P_2.x.append( P_rel.x[-1]*(-P_1.m/(P_1.m+P_2.m)))
    P_2.y.append( P_rel.y[-1]*(-P_1.m/(P_1.m+P_2.m)))
    
    P_1.vX.append( P_rel.vX[-1]*(P_2.m/(P_1.m+P_2.m)))
    P_1.vY.append( P_rel.vY[-1]*(P_2.m/(P_1.m+P_2.m)) )
    P_2.vX.append( P_rel.vX[-1]*(-P_1.m/(P_1.m+P_2.m)))
    P_2.vY.append( P_rel.vY[-1]*(-P_1.m/(P_1.m+P_2.m)))
    
    #NOTA: Il vettore accelerazione sarà di lunghezza N_cycles-1 poichè l'ultima accelerazione non serve perchè non è richiesto aggiornare ulteriormente la posizione e la velocità

In [7]:
initial_conditions(P_1, P_2,P_rel,V_in, r)

In [8]:
j=0
for i in range(0,int (n_cycles),1):
    update(P_1,P_2,P_rel,dt,j)
    j=j+1

In [9]:
def Set_Range(P):
    global Error
    diff_1 = max(P.x)-min(P.x)
    diff_2 = max(P.y)-min(P.y)
    Gap = diff_1-diff_2
    if(Gap<0):
        Error =1
    Rx = max(P.x) + (0.05 * diff_1) # aggiungo un margine del 5% del semiasse maggiore
    rx = min(P.x) - (0.05 * diff_1) # aggiungo un margine del 5% del semiasse maggiore
    Ry = max(P.y) + (Gap/2)
    ry = min(P.y) - (Gap/2)
    return Rx, rx, Ry, ry


In [10]:
#convertion 
ax1=array.array('d',P_1.x)
ay1=array.array('d',P_1.y)
ax2=array.array('d',P_2.x)
ay2=array.array('d',P_2.y)

In [11]:
c1 = ROOT.TCanvas("c1","c1",1000,500)
c1.Divide(2,1)

################## Tela 1 
c1.cd(1)

##### Grafico 1 (disegno un punto ogni 1000 per alleggerire il grafico)
Graph = ROOT.TGraph(len(P_1.x[::1000]),ax1[::1000] , ay1[::1000])
Graph.SetLineColor("kBlue")
Graph.SetLineWidth(2)
Graph.SetMarkerSize(0)
##### Grafico 2 
Graph2 = ROOT.TGraph(len(P_2.x[::1000]),ax2[::1000] , ay2[::1000])
Graph2.SetLineColor('kRed')
Graph2.SetLineWidth(2)
Graph2.SetMarkerColor("kRed")
Graph2.SetMarkerSize(20)

#####MultiGraph
Graph3 = ROOT.TMultiGraph()
Graph3.Add(Graph)
Graph3.Add(Graph2)
Graph3.SetTitle("Orbite dei due corpi;X[m];Y[m]")
T= ROOT.TLegend(0.8, 0.8, 0.9, 0.9)
T.AddEntry(Graph,"m1","l")
T.AddEntry(Graph2,"m2","l")
Graph3.Draw("ALP")
T.Draw()


Rx_1, rx_1, Ry_1, ry_1= Set_Range(P_1)   #Trovo il range giusto per non distorcere l'orbita
Graph3.GetXaxis().SetLimits(rx_1,Rx_1)
Graph3.SetMinimum(ry_1)
Graph3.SetMaximum(Ry_1)

    
################## Tela 2
c1.cd(2)

Rx_2, rx_2, Ry_2, ry_2= Set_Range(P_2)
if (Rx_2-rx_2)!=0 and (Ry_2-ry_2)!=0 :  #serve l'if perchè se la variazione della posizione del sole è troppo piccola (ad esempio mettendo m1=0) allora non si puo impostare il range
    Graph2.GetXaxis().SetLimits(rx_2,Rx_2)
    Graph2.SetMinimum(ry_2)
    Graph2.SetMaximum(Ry_2)

if Error:
    print("ATTENZIONE, errore nel trovare il range corretto dei grafici: \n- L'orbita potrebbe non essere legata.\n- L'orbita potrebbe essere circolare ma con un approssimazione del moto poco precisa. In questo caso, si provi a diminuire l'intervallo dt per calcolare un range corretto.")

Graph2.SetTitle("Orbita del corpo centrale;X[m];Y[m]")
Graph2.Draw("AL")

c1.SaveAs("Orbits.png")
c1.Draw()

#Se non dovesse apparire la tela si guardi il file "Orbits.png"

Info in <TCanvas::Print>: png file Orbits.png has been created


### Moto legato o non legato

E' possibile prevedere se il moto sarà legato a partire dall'eccentricità dell'orbita.
in particolare:
- se e<1 il moto è legato
- se e>1 il moto è NON legato

**Eccentricità**  = $\quad \sqrt{\frac{1+(2Eh^{2})}{\mu^{2}}} \quad$ con $\quad E= \frac{v^2}{2}-\frac{\mu}{r}\quad$ ; $\quad h= \vec{r} \times \vec{v}$ 

In [12]:
def eccentricity(P_1, P_2, P_rel, G):
    mu = G*(P_1.m + P_2.m) 
    
    r = math.sqrt(P_rel.x[0]**2 + P_rel.y[0]**2)
    v_sq = P_rel.vX[0]**2 +  P_rel.vY[0]**2
    
    E = (v_sq / 2.0) - (mu / r)
    
    h = (P_rel.x[0]* P_rel.vY[0])-( P_rel.y[0]* P_rel.vX[0])
    
    # e = sqrt(1+(2*E*h^2)/mu^2)
    e_squared = 1 + (2*E*(h**2)) / (mu**2)
    e = math.sqrt(abs(e_squared))
    
    return e

In [13]:
e = eccentricity(P_1,P_2,P_rel,G)

if (e<1):
    Legata = True
    print(f"L'orbità è legata con eccentricità {e:.3f} \n")

else:
    Legata = False
    print(f"L'orbita NON è legata con eccentricità {e:.3f}")



L'orbità è legata con eccentricità 0.016 



### Confronto con la velocità di orbita circolare

Utilizzo delle soglie convenzionali per identificare il tipo di orbita confrontando la velocità iniziale con la velocità del moto circolare a stessa distanza. Utilizzo:
- Se la velocità differisce per più dell'1% di quella del moto circolare allora l'orbita è ellittica.
- Se la velocità differisce fra lo 0.5% e l'1% da quella di moto circolare allora il moto è "lievemente ellittico".
- Se differisce per meno dello 0.5% allora il moto è approssimabile a orbita circolare. 

In [14]:
V_c= np.sqrt((G*(P_2.m+P_1.m))/r)

if Legata:
    if (V_in > 1.01*V_c):
        print(f"La velocità del corpo è maggiore di quella del moto circolare: l'orbita è di tipo ellittico e la posizione iniziale del corpo 1 corrisponde al perielio.")
    if (V_in < 0.99*V_c):
        print(f"La velocità del corpo è maggiore di quella del moto circolare: l'orbita è di tipo ellittico e la posizione iniziale del corpo 1 corrisponde all'afelio.")
    if (V_in>(0.99*V_c) and V_in<(1.01*V_c)):
        if (V_in>(0.995*V_c)) and (V_in<(1.005*V_c)):
            print("La velocità del corpo è molto simile a quella del moto circolare: l'orbita è approssimabile ad un orbita circolare.")
        else:
            print("La velocità del corpo è simile a quella del moto circolare: l'orbita è di forma lievemente ellittica.")
else:
    print("La velocità è molto maggiore di quella del moto circolare, per cui il sistema NON è legato.")

La velocità del corpo è simile a quella del moto circolare: l'orbita è di forma lievemente ellittica.


## Seconda legge di Keplero

Per verificare la seconda legge di Keplero si può studiare come varia il valore del momento angolare specifico dei due corpi orbitanti nel tempo. Si è scelto di graficare tale valore nel tempo utilizzando la scala logaritmica affinchè si possa visualizzare il momento angolare risultante da diverse simulazioni, ma è possibile adattare il range personalizzandolo alle condizioni iniziali scelte. Ciò che ci si aspetta è che il valore di momento angolare sia pressochè costante, con qualche variazioni dovute alle approssimazioni.

**Momento angolare specifico**:    $ \quad dA=\frac{1}{2} \cdot h \cdot dt \quad \rightarrow \quad \frac{dA}{dt} = \frac{1}{2} \cdot h  \quad ; \quad h= \vec{r} \times \vec{v}$


In [15]:
def Second_Law(P_1,P_2):
    P_1.h=[]
    P_2.h=[]
    P_rel.h=[]
    
    for i in range(0,len(P_1.x)-1,1):
        P_1.h.append(abs(P_1.x[i]*P_1.vY[i] - P_1.y[i]*P_1.vX[i]))
        P_2.h.append(abs(P_2.x[i]*P_2.vY[i] - P_2.y[i]*P_2.vX[i]))
        P_rel.h.append(abs(P_rel.x[i]*P_rel.vY[i] - P_rel.y[i]*P_rel.vX[i]))

In [16]:
if Legata:
    Second_Law(P_1,P_2)
    t=array.array("d",np.linspace(0, (len(P_1.h)-1) * dt, len(P_1.h)))
    h_1=array.array("d",P_1.h)
    h_2= array.array("d",P_2.h)
    h_rel= array.array("d",P_rel.h)

    #Per il corpo 1 rispetto al corpo 2
    c2 = ROOT.TCanvas("c2","c2",1000,1000)
    c2.Divide(2,2)
    c2.cd(1)
    c2.cd(1).SetLogy(True)
    c2.cd(1).SetLeftMargin(0.2)
    Graph5 = ROOT.TGraph(len(P_rel.h[::1000]),t[::1000],h_rel[::1000])
    Graph5.GetYaxis().SetRangeUser(1e8, 5e20)
    Graph5.SetLineWidth(2)
    Graph5.SetLineColor("kGreen")
    Graph5.SetTitle("h(t) per il corpo 1 rispetto al corpo 2;h[m^2/s];t[s]")
    Graph5.Draw("AL")
    
    # Per il corpo P_1 rispetto al CDM
    c2.cd(2)
    c2.cd(2).SetLogy(True)
    c2.cd(2).SetRightMargin(0.2)
    Graph6 = ROOT.TGraph(len(P_1.h[::1000]),t[::1000],h_1[::1000])
    Graph6.GetYaxis().SetRangeUser(1e8, 5e20)
    Graph6.SetLineWidth(2)
    Graph6.SetLineColor("kBlue")
    Graph6.SetTitle("h(t) per il corpo 1 rispetto al CDM;h[m^2/s];t[s]")
    Graph6.Draw("AL")
    
    # Per il corpo P_2 rispetto al CDM
    c2.cd(3)
    c2.cd(3).SetLogy(True)
    c2.cd(3).SetLeftMargin(0.2)
    Graph7 = ROOT.TGraph(len(P_2.h[::1000]),t[::1000],h_2[::1000])
    Graph7.GetYaxis().SetRangeUser(1, 1e10)
    Graph7.SetLineWidth(2)
    Graph7.SetLineColor("kRed")
    Graph7.SetTitle("h(t) per il corpo 2 rispetto al CDM;h[m^2/s];t[s]")
    Graph7.Draw("AL")
    
    c2.Draw("AL")
else:
    raise Exception("Per verificare la seconda legge di Keplero l'orbita dele essere legata!")

## Terza legge di Keplero


 Per verificare la terza legge di keplero è possibile misurare il semiasse maggiore e il periodo di corpi diversi che orbitano intorno allo stesso corpo centrale (in questo caso il sole), per poi confrontare i valori della relativa costante K (il rapporto fra il quadrato del periodo e il cubo del semiasse maggiore) ottenuta dalle simulazioni. Ci aspettiamo che, se l'approssimazione è sufficientemente precisa, i valori di K ottenuti siano simili fra loro (questo solo se il corpo centrale rimane lo stesso per tutti i corpi orbitanti e se esso è molto più massivo).        
                      
ATTENZIONE: Si è assunto il sistema di riferimento del sole come inerziale, poichè la legge di Keplero classica prevede che la massa del corpo due sia molto maggiore di quella del corpo 1!

**Terza legge di Keplero** : $ \quad K=\frac{T^{2}}{a^{3}}$

In [19]:
def Third_Law(P_1):
    T=[]
    for i in range (0,n_cycles-1,1):
        if(P_1.vX[i]>0 and P_1.vX[i+1]<0): #Trovo il valore dell'indice i corrispondente al'inversione di moto lungo x 
            T.append(dt*i)
    a= (max(P_rel.x) - min(P_rel.x))/2
    k= pow(T[0],2) / pow(a,3)
    print(f"\n- Semiasse maggiore: {(a/1000):.3f} km  - ({(a/(1.496e11)):.2f} u.a)\n")
    print(f"- Periodo: {T[0]} s - ({(T[0]/(60*60*24*365)):.2f} anni terrestri) \n")
    print(f"- Costante K di Keplero: {(k*pow(10,19)):.2f} e-19  (s^2)/(m^3)\n")

In [20]:
if Legata:
    Third_Law(P_1)
else:
    raise Exception("Per verificare la terza legge di Keplero l'orbita dele essere legata!")


- Semiasse maggiore: 149743154.852 km  - (1.00 u.a)

- Periodo: 31609350 s - (1.00 anni terrestri) 

- Costante K di Keplero: 2.98 e-19  (s^2)/(m^3)



### Risultati ottenuti per la costante K del sole (in s<sup>2</sup>/m<sup>3</sup>) 

- **Mercurio** : 2.98 e-19
- **Terra** : 2.98 e-19
- **Marte** : 2.98 e-19
- **Giove** : 2.97 e-19  
- **Cometa** : 2.95 e-19

Si può provare ad ottenere K per un qualsiasi corpo orbitante al sole cambiando le condizioni iniziali.